# Day 22: Multimodal Embeddings

One model. Text, images, and audio — all embedded together.

Google's new `gemini-embedding-2-preview` model creates embeddings for any content type.

## Setup

In [30]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv
import numpy as np

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Text Embeddings

In [31]:
# Embed text
result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=["What is the meaning of life?"]
)

print(f"Embedding dimension: {len(result.embeddings[0].values)}")
print(f"First 10 values: {result.embeddings[0].values[:10]}")

Embedding dimension: 3072
First 10 values: [-0.016332133, -0.0043764366, -0.0011324773, -0.011240026, 0.00029597108, 0.0014934157, -0.018807797, 0.013529229, 0.019723475, -0.049145423]


## Image Embeddings

In [32]:
# Read image
with open("image.png", "rb") as f:
    image_bytes = f.read()

# Embed image
result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=[
        types.Part.from_bytes(
            data=image_bytes,
            mime_type="image/png",
        )
    ]
)

print(f"Image embedding dimension: {len(result.embeddings[0].values)}")

Image embedding dimension: 3072


## Cross-Modal Similarity

The magic: compare text to images using cosine similarity!

In [33]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Embed text descriptions
text_result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=[
        "A cat sitting on a couch",
        "A dog running in a park",
        "A mountain landscape"
    ]
)

# Embed image
with open("sample.png", "rb") as f:
    image_bytes = f.read()

image_result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/png")
    ]
)

# Compare image to each text description
image_embedding = image_result.embeddings[0].values
descriptions = ["A cat lying on a grass", "A dog running in a park", "A mountain landscape"]

print("Image-to-text similarity:")
for i, desc in enumerate(descriptions):
    text_embedding = text_result.embeddings[i].values
    sim = cosine_similarity(image_embedding, text_embedding)
    print(f"  '{desc}': {sim:.4f}")

Image-to-text similarity:
  'A cat lying on a grass': 0.3441
  'A dog running in a park': 0.2275
  'A mountain landscape': 0.2182


## Audio Embeddings

In [ ]:
# Read audio (if you have a sample audio file)
with open("sample.mp3", "rb") as f:
    audio_bytes = f.read()

# Embed audio
result = client.models.embed_content(
    model="gemini-embedding-2-preview",
    contents=[
        types.Part.from_bytes(
            data=audio_bytes,
            mime_type="audio/mpeg",
        )
    ]
)

print(f"Audio embedding dimension: {len(result.embeddings[0].values)}")

## Key Takeaways

1. **One model, all modalities** — Text, images, audio in the same embedding space
2. **Cross-modal search** — Search images with text, or find similar audio to an image
3. **Same API** — Just use `types.Part.from_bytes()` for non-text content
4. **Same dimension** — All embeddings have the same dimension, enabling comparison

**Model:** `gemini-embedding-2-preview`

---

**Use cases:**
- Image search with natural language
- Multimodal RAG (retrieve images based on text queries)
- Content similarity across modalities
- Audio/video search